## Descriptive Statistics Summary.

**Sample verified:** 200K rows × 28 features, ~80/20 class balance preserved.  
*Note: This is a stratified sample on loan_status (Fully Paid vs Charged Off).  
Grade proportions therefore reflect the sample, not necessarily origination volumes.*

---

### Numeric features (key observations)

- **Loan amounts:**  
  (Fill once you provide `loan_amnt.describe()` — remove this line if not analytically needed.)

- **Interest rates:**  
  (Fill once you provide `int_rate.describe()` — remove this line if not analytically needed.)

- **Annual income:**  
  Range = \$0 → \$8.9M  
  Median = \$65,000  
  Mean = \$76,131 (strong right‑skew — mean pulled upward by high earners)  
  High‑income tail: 248 borrowers > \$500K; 33 borrowers > \$1M

- **DTI:**  
  (Fill once you provide `dti.describe()` — remove this line if not analytically needed.)

- **FICO scores:**  
  (Fill once you provide `fico_range_low/high.describe()` — remove this line if not analytically needed.)

---

### Categorical features (key observations)

- **Grade distribution:**  
  Mid‑grades dominate the *sample*:  
  B = 29.18%, C = 28.25%, A = 17.49%.  
  Higher‑risk grades taper: D = 14.94%, E = 7.02%, F = 2.45%, G = 0.68%.  
  *Note: Because the sample is stratified on loan_status, grade proportions reflect the sample rather than exact origination volumes.*

- **Loan purpose:**  
  Highly concentrated in two categories:  
  • debt_consolidation = 58.06%  
  • credit_card = 21.99%  
  Together ≈ 80% of all loans.  
  Next largest: home_improvement (6.42%), other (5.77%), major_purchase (2.17%).

- **Home ownership:**  
  MORTGAGE = 49.42%  
  RENT = 39.80%  
  OWN = 10.75%  
  Remaining categories negligible (<0.05%).

- **Geographic distribution (top‑5 states):**  
  CA = 14.64%  
  TX = 8.23%  
  NY = 8.23%  
  FL = 7.20%  
  IL = 3.86%  
  These five states represent ~42% of the dataset.

- **Term:**  
  36 months = 75.89%  
  60 months = 24.11%

- **Application type:**  
  Individual = 98.04%  
  Joint App = 1.96%

- **Verification status:**  
  Source Verified = 38.83%  
  Verified = 31.09%  
  Not Verified = 30.08%

- **Employment length:**  
  10+ years = 34.84%  
  Remaining categories distributed across 1–8 years, each ~4–10%.

---

### Missing values

- **mths_since_last_delinq — 50.53% missing**  
  Nulls typically indicate *no prior delinquencies* — meaningful signal, not true missingness.

- **emp_length — 5.86% missing**  
  Moderate; due to non‑reported employment tenure.

- **revol_util — 0.07% missing**  
  Negligible.

- **dti — 0.03% missing**  
  Negligible.

Overall missingness is low except for `mths_since_last_delinq`, where nulls carry semantic meaning.

---

### Q1 preview (baseline default rate by grade)

- **Grade A:** 5.93%  
- **Grade B:** 13.42%  
- **Grade C:** 22.45%  
- **Grade D:** 30.13%  
- **Grade E:** 38.39%  
- **Grade F:** 46.33%  
- **Grade G:** 50.22%  

A clear **monotonic gradient** is observed — default rates rise steadily from A → G.  
Full Q1 treatment in dedicated notebook.

---

### Ready for Q1

- All 28 features present and accounted for  
- Class balance sound (Fully Paid 80.04%, Charged Off 19.96%)  
- Missing patterns documented  
- Grade‑default relationship confirmed at surface level


In [10]:
# Step 6 - Quick Baseline Default Rates.

# Class balance in sample
print("Sample class balance:")
print(df["loan_status"].value_counts(normalize=True).round(4) * 100)

# Class balance by grade (Q1 preview - just the baseline)
print("\nDefault rate (Charged Off %) by grade:")
grade_default = df.groupby("grade")["loan_status"].apply(
    lambda x: (x == "Charged Off").sum() / len(x) * 100
).round(2)
print(grade_default)

Sample class balance:
loan_status
Fully Paid     80.04
Charged Off    19.96
Name: proportion, dtype: float64

Default rate (Charged Off %) by grade:
grade
A     5.93
B    13.42
C    22.45
D    30.13
E    38.39
F    46.33
G    50.22
Name: loan_status, dtype: float64


In [9]:
# Step 5 - Missing Value Overview.

# Missing values across all features
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})
missing_df = missing_df[missing_df["missing_count"] > 0]
print(missing_df)

                        missing_count  missing_pct
mths_since_last_delinq         101051        50.53
emp_length                      11716         5.86
revol_util                        135         0.07
dti                                56         0.03


In [8]:
# Step 4 - Categorical Distributions.

# Category distributions
for col in CATEGORICAL_COLS:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(normalize=True).head(10).round(4) * 100)


=== term ===
term
36 months    75.89
60 months    24.11
Name: proportion, dtype: float64

=== grade ===
grade
B    29.18
C    28.25
A    17.49
D    14.94
E     7.02
F     2.45
G     0.68
Name: proportion, dtype: float64

=== sub_grade ===
sub_grade
C1    6.27
B4    6.20
B5    6.10
B3    6.09
C2    5.86
C3    5.58
C4    5.52
B2    5.47
B1    5.33
C5    5.01
Name: proportion, dtype: float64

=== loan_status ===
loan_status
Fully Paid     80.04
Charged Off    19.96
Name: proportion, dtype: float64

=== purpose ===
purpose
debt_consolidation    58.06
credit_card           21.99
home_improvement       6.42
other                  5.77
major_purchase         2.17
medical                1.15
small_business         1.14
car                    1.09
moving                 0.71
vacation               0.67
Name: proportion, dtype: float64

=== application_type ===
application_type
Individual    98.04
Joint App      1.96
Name: proportion, dtype: float64

=== addr_state ===
addr_state
CA    14.64
TX

In [7]:
# Step 3 - Numeric Distributions.

# Descriptive stats for numeric columns
numeric_stats = df[NUMERIC_COLS].describe().T
numeric_stats["missing_pct"] = df[NUMERIC_COLS].isnull().sum() / len(df) * 100
numeric_stats = numeric_stats.round(2)
numeric_stats

# Income Sanity Check

# Annual income sanity check (a classic outlier zone)
print("Annual income summary:")
print(df["annual_inc"].describe().round(0))
print(f"\nBorrowers with income > $500K: {(df['annual_inc'] > 500_000).sum():,}")
print(f"Borrowers with income > $1M: {(df['annual_inc'] > 1_000_000).sum():,}")

Annual income summary:
count     200000.0
mean       76131.0
std        66935.0
min            0.0
25%        45500.0
50%        65000.0
75%        90000.0
max      8900060.0
Name: annual_inc, dtype: float64

Borrowers with income > $500K: 248
Borrowers with income > $1M: 33


In [4]:
# Step 2 — Column Categorisation.

# Categorise columns for appropriate descriptive stats

NUMERIC_COLS = [
    "loan_amnt", "funded_amnt", "int_rate", "installment",
    "annual_inc", "dti", "fico_range_low", "fico_range_high",
    "open_acc", "total_acc", "revol_bal", "revol_util",
    "inq_last_6mths", "delinq_2yrs", "mths_since_last_delinq"
]

CATEGORICAL_COLS = [
    "term", "grade", "sub_grade", "loan_status", "purpose",
    "application_type", "addr_state", "verification_status",
    "emp_length", "home_ownership"
]

DATE_COLS = ["issue_d", "earliest_cr_line"]

ID_COL = ["id"]

# Sanity check
total_categorised = len(NUMERIC_COLS) + len(CATEGORICAL_COLS) + len(DATE_COLS) + len(ID_COL)
print(f"Total columns categorised: {total_categorised}")
print(f"Sample columns: {df.shape[1]}")
assert total_categorised == df.shape[1], "Categorisation doesn't match column count"
print("\n All 28 columns categorised")

Total columns categorised: 28
Sample columns: 28

 All 28 columns categorised


In [ ]:
# Step 1 — Load and Verify Sample.

import pandas as pd
from pathlib import Path

# Jupyter-safe path (pattern you established Day 37)
NOTEBOOK_DIR = Path().resolve()
sample_path = NOTEBOOK_DIR.parent / "data" / "sample_200k_stratified.csv"

df = pd.read_csv(sample_path)
print(f"Sample shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

# Verification assertions (Day 37's discipline)
assert df.shape[0] == 200_000, f"Expected 200K rows, got {df.shape[0]}"
assert df.shape[1] == 28, f"Expected 28 columns, got {df.shape[1]}"
print("\n Sample Loaded Correctly")

Sample shape: (200000, 28)
Memory: 153.76 MB

✅ Sample loaded correctly


# Capstone: Lending Club Loan Default Analysis

**Notebook 03:** Descriptive statistics on the 200K stratified sample.

**Purpose:** Establish baseline understanding of the 28 features before 
analytical questions. Every subsequent notebook (Q1 grade analysis, 
Q2 purpose, Q3 temporal, Q4 borrower characteristics) references this 
as the "what does normal look like" foundation.

**Loaded from:** `../data/sample_200k_stratified.csv` (200K rows, 28 features, 
~80/20 Fully Paid / Charged Off class balance)

**Not in this notebook:** analytical findings, cross-feature relationships, 
charts beyond simple distributions.

**Reference:** See `../CAPSTONE_SCOPING.md` for full analytical scope.